# Pipeline 2 - Limitar abstracts ATUALIZADA 40000

Entrada: `arxiv_amostra_40000_multilabel.json`. Saida: `arxiv_amostra_40000_abstracts_limitados_atualizada.json`.


## 1. Configuracao

In [ ]:
N_FRASES   = 3        # no de frases mantidas por abstract
LAMBDA_CAT = 0.5      # peso do sinal de categoria (0 = so paper; 1 = paper+categoria iguais)

# token: palavras de letras (e hifen), com 3+ caracteres
TOKEN_RE = r"(?u)\b[a-zA-Z][a-zA-Z-]{2,}\b"

print(f"Limite: {N_FRASES} frases/abstract | peso categoria = {LAMBDA_CAT}")

## 2. Montar o Drive e carregar a amostra (JSON Lines)

In [ ]:
from google.colab import drive
import os, pandas as pd
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/projetoIA-EquipeLoremIpsum (1)'  # opcional: coloque a pasta exata do projeto
NOME_IN = 'arxiv_amostra_40000_multilabel.json'

def shallow_find(root, filename, max_depth=3):
    root = os.path.abspath(root)
    root_depth = root.rstrip(os.sep).count(os.sep)
    for current, dirs, files in os.walk(root):
        depth = current.rstrip(os.sep).count(os.sep) - root_depth
        if depth >= max_depth:
            dirs[:] = []
        if filename in files:
            return os.path.join(current, filename)
    return None

candidatos = []
if BASE:
    candidatos.extend([os.path.join(BASE, NOME_IN), os.path.join(BASE, 'pipelines', NOME_IN)])
candidatos.extend([os.path.join('/content/drive/MyDrive', NOME_IN), os.path.join('/content/drive/MyDrive', 'pipelines', NOME_IN)])

CAMINHO_IN = next((p for p in candidatos if os.path.exists(p)), None)
if CAMINHO_IN is None:
    CAMINHO_IN = shallow_find('/content/drive/MyDrive', NOME_IN, max_depth=3)
if CAMINHO_IN is None:
    raise FileNotFoundError(NOME_IN)

PIPE = os.path.dirname(CAMINHO_IN)
SAIDA = os.path.join(PIPE, 'arxiv_amostra_40000_abstracts_limitados_atualizada.json')
print('Input file:', CAMINHO_IN)
print('Output file:', SAIDA)

df = pd.read_json(CAMINHO_IN, lines=True)
print('Carregado:', df.shape)
print('Colunas:', list(df.columns))
df[['id', 'title', 'primary_category']].head()


## 3. TF-IDF  nivel documento e nivel categoria

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# (1) TF-IDF por documento (cada abstract e um documento)
vec_doc = TfidfVectorizer(stop_words='english', token_pattern=TOKEN_RE, lowercase=True)
X_doc   = vec_doc.fit_transform(df['abstract']).tocsr()
vocab_d = vec_doc.get_feature_names_out()

# (2) TF-IDF por categoria (todos os abstracts da categoria concatenados = 1 documento)
cat_text = df.groupby('primary_category')['abstract'].apply(' '.join)
vec_cat  = TfidfVectorizer(stop_words='english', token_pattern=TOKEN_RE, lowercase=True)
X_cat    = vec_cat.fit_transform(cat_text.values).tocsr()
vocab_c  = vec_cat.get_feature_names_out()

# pre-calcula os scores de categoria como dicionarios {palavra: peso}
cat_pos    = {c: i for i, c in enumerate(cat_text.index)}
cat_scores = {}
for c, i in cat_pos.items():
    row = X_cat.getrow(i)
    cat_scores[c] = {vocab_c[j]: v for j, v in zip(row.indices, row.data)}

print('Vocabulario (documentos):', len(vocab_d))
print('Vocabulario (categorias):', len(vocab_c))


## 4. Reduzir os abstracts (extracao das N frases mais relevantes)

In [ ]:
import re
import nltk

# divisor de frases do NLTK
for pkg in ['punkt', 'punkt_tab']:
    try:
        nltk.data.find(f'tokenizers/{pkg}')
    except LookupError:
        nltk.download(pkg, quiet=True)
from nltk.tokenize import sent_tokenize

# --- edge case: abreviacoes com ponto no meio que o NLTK divide errado ---
# protegemos o ponto com um sentinela (U+2024, ONE DOT LEADER) antes de dividir
# e restauramos depois. Ex.: "i.i.d." nao vira fim de frase.
_ABREV = ['i.i.d.', 'e.g.', 'i.e.', 'et al.', 'al.', 'etc.', 'vs.', 'cf.', 'resp.',
          'w.r.t.', 'a.k.a.', 'Fig.', 'Eq.', 'Eqs.', 'Ref.', 'Refs.', 'approx.',
          'Dr.', 'Prof.', 'No.', 'Inc.', 'Sec.', 'Thm.', 'Eq.', 'Sr.', 'St.']
_SENT = '.'   # sentinela que substitui o '.' temporariamente

def _proteger(t):
    for a in _ABREV:
        t = t.replace(a, a.replace('.', _SENT))
    return t

def dividir_frases(texto):
    texto = ' '.join(texto.split())                 # normaliza quebras de linha/espacos
    texto = _proteger(texto)                         # blinda as abreviacoes
    try:
        frases = sent_tokenize(texto)
    except Exception:                                # fallback: split simples por pontuacao
        frases = re.split(r'(?<=[.!?])\s+(?=[A-Z])', texto)
    frases = [f.replace(_SENT, '.').strip() for f in frases]   # restaura os pontos
    return [f for f in frases if f]

def score_frase(frase, doc_s, cat_s):
    # palavras unicas da frase (evita que repeticao infle o score)
    palavras = {w.lower() for w in re.findall(TOKEN_RE, frase)}
    return sum(doc_s.get(w, 0.0) + LAMBDA_CAT * cat_s.get(w, 0.0) for w in palavras)

def reduzir_abstract(i, abstract, categoria):
    """Mantem as top-N frases mais relevantes (paper + categoria), na ordem original."""
    row   = X_doc.getrow(i)
    doc_s = {vocab_d[j]: v for j, v in zip(row.indices, row.data)}
    cat_s = cat_scores.get(categoria, {})

    frases = dividir_frases(abstract)
    if len(frases) <= N_FRASES:
        return ' '.join(frases)                      # ja e curto: mantem tudo

    pont = [(k, score_frase(f, doc_s, cat_s)) for k, f in enumerate(frases)]
    top  = sorted(pont, key=lambda x: x[1], reverse=True)[:N_FRASES]
    idx  = sorted(k for k, _ in top)                 # volta a ordem original do abstract
    return ' '.join(frases[k] for k in idx)

df['abstract_reduzido'] = [
    reduzir_abstract(i, ab, cat)
    for i, (ab, cat) in enumerate(zip(df['abstract'], df['primary_category']))
]

# estatisticas de reducao
df['n_palavras_orig']     = df['abstract'].str.split().str.len()
df['n_palavras_reduzido'] = df['abstract_reduzido'].str.split().str.len()
df['n_frases_orig']       = df['abstract'].map(lambda t: len(dividir_frases(t)))
df['n_frases_reduzido']   = df['abstract_reduzido'].map(lambda t: len(dividir_frases(t)))
print('Frases  (orig)   media:', round(df['n_frases_orig'].mean(), 1))
print('Palavras(orig)   media:', round(df['n_palavras_orig'].mean(), 1))
print('Palavras(reduzido) media:', round(df['n_palavras_reduzido'].mean(), 1))
print('Reducao media: {:.0%}'.format(1 - df['n_palavras_reduzido'].sum() / df['n_palavras_orig'].sum()))


In [ ]:
# Exemplo antes/depois
ex = df.iloc[0]
print('CATEGORIA:', ex['primary_category'])
print('TITULO   :', ex['title'])
print('\n--- ORIGINAL ---\n', ex['abstract'])
print('\n--- REDUZIDO ---\n', ex['abstract_reduzido'])


## 5. Salvar como JSON Lines e baixar para o PC

Mesma logica da Pipeline 1: salva no Drive (pasta `pipelines`) e baixa para a sua maquina. Mantem todos os campos originais (titulo e categorias **intactos**) e adiciona `abstract_reduzido` (as N frases mais relevantes). Use o campo `abstract_reduzido` como entrada do BERT na Pipeline 3.

In [ ]:
df.to_json(SAIDA, orient='records', lines=True, force_ascii=False)
print('Salvo:', SAIDA, '|', len(df), 'registros (JSON Lines)')

try:
    from google.colab import files
    files.download(SAIDA)
except Exception as e:
    print('Download automatico indisponivel (rodando fora do Colab?):', e)
